# 🛠️ Notebook: Introduction to Gradio

This notebook covers the most important basics of Gradio.

## 📚 Sources

- [Gradio Docs](https://gradio.app/docs/)

---

Good luck experimenting with Gradio! 🤗

In [1]:
import gradio as gr

/Users/nils_hellwig/Desktop/Projects/ai-engineering-notebooks/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
gr.__version__

'6.20.0'

### Example 1: Simple User Interface (Example from the slides)

In Gradio, we can create a user interface (UI) for our LLM applications with just a few lines of code.
Unlike web frameworks such as Flask or FastAPI, we don't need to worry about web development details but can focus on functionality.

The simplest way to create a UI is by using the `gr.Interface` class.
`gr.Interface` requires at least three arguments:


```python

- `fn`:  The function that is called when the user submits the inputs.
  The function around which a user interface (UI) is built.

- `inputs`:  
  The Gradio component(s) used as input.  
  👉 The number of components must match the number of arguments in your function.  

- `outputs`:  
  The Gradio component(s) used as output.  
  👉 The number of components must match the number of return values from your function.  
```

In [3]:
def greet(name, intensity):
    return "Hello, " + name + "!" * intensity

demo = gr.Interface(
    fn=greet,
    inputs=["text", "slider"],
    outputs=["text"],
)

# With share=False, no link is generated to share the app publicly (!). 
# Note: By default, share=False, which means no link is generated to share the app publicly.
demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### Example 2: User Interface with various components

You can find documentation of all components here: [Gradio Components](https://www.gradio.app/docs/gradio)



**Function `sentence_builder`:**
Generates an English sentence from various inputs using f-string and `join()`.

**Interface Components:**

- **Slider**: Number range (2-20) with default value
- **Dropdown**: Single selection for animals
- **CheckboxGroup**: Multiple selection for countries
- **Radio**: Single selection for locations
- **Dropdown (multiselect)**: Multiple selection for activities with preset
- **Checkbox**: Yes/No for time of day

**Key Features**

- `info`: Additional description below the label
- `value`: Set default values
- `multiselect=True`: Enable multiple selection
- `examples`: Pre-made test data for quick demo

**Output**

Example: "The 4 cats from Japan and Pakistan went to the park where they ate and swam until the morning"

In [4]:
# The order of parameters in the function must match the order of inputs
def sentence_builder(quantity, animal, countries, place, activity_list, morning):
    return f"""The {quantity} {animal}{"s" if quantity > 1 else ""} from {" and ".join(countries)} went to the {place} where they {" and ".join(activity_list)} until the {"morning" if morning else "night"}"""

demo = gr.Interface(
    fn=sentence_builder,
    inputs=[
        # The info is displayed below the label
        gr.Slider(2, 20, value=4, label="Count",
                  info="Choose between 2 and 20"),
        gr.Dropdown(["cat", "dog", "bird"], label="Animal",
                    info="Will add more animals later!"),
        gr.CheckboxGroup(["USA", "Japan", "Pakistan"],
                         label="Countries", info="Where are they from?"),
        gr.Radio(["park", "zoo", "road"], label="Location",
                 info="Where did they go?"),
        gr.Dropdown(["ran", "swam", "ate", "slept"],
                    # Default selection for the dropdown
                    value=["swam", "slept"],
                    multiselect=True,
                    label="Activity",
                    info="Lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed auctor, nisl eget ultricies aliquam, nunc nisl aliquet nunc, eget aliquam nisl nunc vel nisl."
                    ),
        gr.Checkbox(label="Morning", info="Did they do it in the morning?"),
    ],
    outputs=gr.Textbox(
        label="Output", info="This is the generated sentence based on your inputs."),  # Output format
    examples=[
        # Using examples, we can provide pre-made inputs that the user can select to test the app
        [2, "cat", ["Japan", "Pakistan"], "park", ["ate", "swam"], True],
        [4, "dog", ["Japan"], "zoo", ["ate", "swam"], False],
        [10, "bird", ["USA", "Pakistan"], "road", ["ran"], False],
        [8, "cat", ["Pakistan"], "zoo", ["ate"], True],
    ]
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


### Example 3: Blocks

1. `Blocks`

- `gr.Blocks` is the container layout system in Gradio, allowing complex user interfaces to be built modularly.  
- Within a Blocks, various layout elements such as `Row`, `Column`, or Tabs can be defined.  
- You can also assign themes (e.g., `gr.themes.Citrus()`) to control the design globally.  
- `with` in Python (context manager): is used as a **context manager** to open a specific scope for objects. In Gradio, this means: All components written within `with gr.Row():` or `with gr.Column():` automatically belong to this layout. This saves explicit assignments and makes code more readable since the hierarchy is clearly visible through indentation.  


---

2. `Row`

- `gr.Row()` arranges the contained components **side by side in a horizontal row**.  
- All elements within a Row share the available space equally (unless specific widths are specified).  
- Typical example: Multiple input fields such as textboxes or number fields that should appear side by side.  

---

3. `Column`

- `gr.Column()` arranges the contained components **one below another in a vertical column**.  
- It's suitable for displaying content in a structured way, e.g., a button and its corresponding output field.  
- You can also nest `Row` and `Column` to create flexible layouts (e.g., a row with columns that in turn contain rows).

In [5]:
import gradio as gr

# Function that is executed when the button is clicked
def greeting(firstname, lastname, age):
    return f"Hello, {firstname} {lastname}! You are {age} years old."

# Create the block
with gr.Blocks() as demo:
    # Vertical layout
    with gr.Row():
        firstname_input = gr.Textbox(label="First Name")
        lastname_input = gr.Textbox(label="Last Name")
        age_input = gr.Number(label="Age")
    with gr.Column():
        greeting_button = gr.Button("Say Hello")
        output_text = gr.Textbox(label="Output")

    # Event: Button clicks -> Execute function -> Write result to output_text (More about events directly below)
    greeting_button.click(greeting, 
                         inputs=[firstname_input, 
                                 lastname_input, 
                                 age_input], 
                         outputs=output_text)

# demo.launch(theme=gr.themes.Citrus())

### Example 4: Events

#### A) Click Event

👉 Explanation: Through `.click()`, a function `fn` is called. `.click()` is supported by `Button`, `ClearButton`, and `UploadButton`.

In [6]:
import gradio as gr

# Function that is called when the button is clicked
def greet(name):
    return f"Hello {name}!"

with gr.Blocks() as demo:
    # Input field for the name
    name = gr.Textbox(label="Enter name")
    
    # Output field for the response
    output = gr.Textbox(label="Response")
    
    # Button that triggers the event
    btn = gr.Button("Greet")
    
    # Event: When the button is clicked (.click),
    # the function greet() is executed.
    # Inputs = "name", Outputs = "output"
    btn.click(fn=greet, inputs=name, outputs=output)

# demo.launch()

#### B) Submit Event

👉 Explanation: Through `.submit()`, a function `fn` is called as soon as the user presses Enter or submits a form. `.submit()` is supported by `Textbox` and `Chatbot`.

In [7]:
import gradio as gr

# Function: returns the text backwards
def reverse_text(text):
    return text[::-1]

with gr.Blocks() as demo:
    # Input field: User types text and presses Enter
    inp = gr.Textbox(label="Enter text and press Enter")
    
    # Output: shows the reversed text
    out = gr.Textbox(label="Reversed text")
    
    # Event: When the user presses Enter (.submit),
    # the function reverse_text() is executed
    inp.submit(fn=reverse_text, inputs=inp, outputs=out)

# demo.launch()

#### C) Change Event

👉 Explanation: Through `.change()`, a function `fn` is called as soon as the value of an element changes. `.change()` is supported by `Dropdown`, `Checkbox`, `CheckboxGroup`, `Radio`, `Slider`, `Number`, `ColorPicker`, `Image`, and `File`.

In [8]:
import gradio as gr

# Function: describes the selected color
def describe_color(color):
    return f"You selected {color}."

with gr.Blocks() as demo:
    # Dropdown with three options
    dropdown = gr.Dropdown(choices=["Red", "Green", "Blue"], label="Choose color")
    
    # Output: shows the description
    out = gr.Textbox(label="Description")
    
    # Event: When the selection in the dropdown is changed (.change),
    # the function describe_color() is executed
    dropdown.change(fn=describe_color, inputs=dropdown, outputs=out)

# demo.launch()

#### D) Input Event

👉 Explanation: Through `.input()`, a function `fn` is called during input into the element (e.g., with each keystroke). `.input()` is supported by `Textbox`, `Number`, and `Slider`.

In [9]:
import gradio as gr

# Function: calculates the length of the entered text
def live_length(text):
    return f"Length: {len(text)} characters"

with gr.Blocks() as demo:
    # Input field where you type live
    inp = gr.Textbox(label="Type something...")
    
    # Output: shows the text length immediately
    out = gr.Textbox(label="Length")
    
    # Event: While the user types (.input),
    # the function live_length() is called immediately
    inp.input(fn=live_length, inputs=inp, outputs=out)

# demo.launch()

#### E) Chaining with `.then()`

👉 Explanation: Through `.then()`, a function `fn` is called after another function (e.g., through `.click()` or `.submit()`) has been successfully executed. `.then()` is supported by all elements that previously trigger an event like `.click()`, `.submit()`, or `.input()`.

In [10]:
import gradio as gr

# Function 1: Greets the user
def greet(name):
    return f"Hello {name}!"

# Function 2: Returns the length of the name
def name_length(greeting):
    return f"Greeting length: {len(greeting)} characters"

with gr.Blocks() as demo:
    name = gr.Textbox(label="Enter name")
    output1 = gr.Textbox(label="Greeting")
    output2 = gr.Textbox(label="Greeting length")
    btn = gr.Button("Greet")
    
    # Event: first call greet(), then name_length() with .then()
    btn.click(fn=greet, inputs=name, outputs=output1).then(fn=name_length, inputs=output1, outputs=output2)

# demo.launch()

### Example 5: Connecting layouts and scaling

In [11]:
import gradio as gr

with gr.Blocks() as demo:
    with gr.Row():
        with gr.Column(scale=1):
            gr.Textbox(label="Left 1")
            gr.Textbox(label="Left 2")
        with gr.Column(scale=2):
            gr.Textbox(label="Right 1")
            gr.Textbox(label="Right 2")

# demo.launch()

### Example 6: Tabs

Tab() is a layout element. Components defined within the tab are visible when this tab is selected.

In [12]:
with gr.Blocks() as demo:
    with gr.Tab("Tab 1"):
        gr.Textbox(label="Text 1")
        gr.Textbox(label="Text 2")
    with gr.Tab("Tab 2"):
        gr.Textbox(label="Text 1")
        gr.Textbox(label="Text 2")
        
# demo.launch()

### Example 7: Chatbot

In [13]:
history = [
    {"role": "user", "content": "What time is it?"},
    {"role": "assistant", "content": "It's 3 PM."},
]

with gr.Blocks() as demo:
    gr.Chatbot(history)

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


### Example 8: Chatbot with input field and button

In this example, a complete **chatbot** is implemented that communicates with a Large Language Model (LLM). The example shows several important concepts:

- **`gr.Chatbot`**: A special component for displaying chat histories in typical messenger style. It expects the OpenAI-compatible message format (`{"role": "...", "content": "..."}`), which is exactly what an OpenAI-compatible chat API returns - no conversion needed.

- **`gr.Row()` and `gr.Column()`**: Enable flexible layout, here to arrange the input field and send button side by side. The `scale` parameter controls the relative width of the columns.

- **`.then()` chaining**: This method allows multiple functions to be executed sequentially. Here, the user message is added first (`add_user_message`), and then the LLM response is automatically fetched (`get_llm_response`).

- **OpenAI Client**: Communication with the LLM is done via the OpenAI-compatible API, which allows the code to also work with local LLMs (like here via Ollama). Same client setup as `04_2_intro_structured_outputs.ipynb`.

In [14]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads LLM_HOST from a .env file in the project root (see notebook 03 / setup.md)

LLM_HOST = os.environ["LLM_HOST"]  # the IP address you got in the lecture
LLM_URL = f"http://{LLM_HOST}:11434/v1"
LLM_REASONING = "gemma4:26b"  # the reasoning MoE model from chapter 06

In [15]:
import openai

client = openai.OpenAI(
    base_url=LLM_URL,
    api_key="ollama",
)

history = [
    {"role": "user", "content": "What time is it?"},
    {"role": "assistant", "content": "It's 3 PM."},
]


def add_user_message(message, chat_history):
    """Immediately adds the user message"""
    if message.strip():
        chat_history.append({"role": "user", "content": message})
    return chat_history, ""


def get_llm_response(chat_history):
    """Fetches the LLM response and adds it"""
    if not chat_history or chat_history[-1]["role"] != "user":
        return chat_history

    # API request to the LLM using OpenAI client
    response = client.chat.completions.create(
        model=LLM_REASONING,
        reasoning_effort="none",  # a snappy chat reply doesn't need extended thinking
        messages=chat_history,
        stream=False
    )

    assistant_reply = response.choices[0].message.content
    chat_history.append({"role": "assistant", "content": assistant_reply})

    return chat_history


with gr.Blocks() as demo:
    chatbot = gr.Chatbot(history)
    with gr.Row():
        with gr.Column(scale=4):
            user_input = gr.Textbox(
                label="User Input", placeholder="Type a message...")
        with gr.Column(scale=1):
            send_btn = gr.Button("Send", variant="primary")

    # Chain the functions: first add user message, then fetch LLM response
    # .then() in Gradio chains two functions so that the second is automatically executed after the first completes.
    # The outputs of the first function are passed as inputs to the second function, creating sequential processing.
    send_btn.click(
        fn=add_user_message,
        inputs=[user_input, chatbot],
        outputs=[chatbot, user_input]
    ).then(
        fn=get_llm_response,
        inputs=[chatbot],
        outputs=[chatbot]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## Task 1: Aspect-Based Sentiment Analysis (ABSA) with Gradio


In this task, you will implement a small **Aspect-Based Sentiment Analysis (ABSA)** application with Gradio.
ABSA means that the sentiment (positive, neutral, negative) is recognized in relation to specific aspects in a text.

The goal is for the user to enter a text and a list of categories in the UI. The application should then recognize the sentiment (😊 positive, 😐 neutral, 😞 negative) for each category in the text and display it in a table in the UI.

---

### Steps

1. **Validate inputs**  
   - If no text: return error message.  
   - If no category(ies): return error message.  
   - Note: Categories must be entered **comma-separated**.

2. **Create prompt & call model**  
   - Use the function `get_user_prompt(text, categories_list)`.  
   - Call the LLM with `llm_text_to_json(user_prompt)`.  
   - Result is a JSON with `items: [{category, sentiment}]`.

3. **Format output**  
   - Create a table with columns: **Category**, **Sentiment**.  
   - If no category recognized → display note.

4. **Build UI**  
   - Use Gradio `Blocks` with:
     - Textbox for the text  
     - Textbox for categories  
     - Button "Analyze sentiment"  
     - Table with results  
     - Text for error messages  
     - Examples (`gr.Examples`) for testing

---

## 💡 Tips
- Use `strip()` and `split(",")` to clean categories.  
- Use `if not ...:` for validations.  
- Return value of `analyze_sentiment_guided`:  
  ```python
  return table_data, error_message
  ```
- Consider the empty array case: No category recognized → display note.

The following code is given:

In [16]:
# First, let's define the prompt function
def get_user_prompt(text, categories_list):
    user_prompt = f"""Text: {text}
Categories: {categories_list}

You are an expert in Aspect-Based Sentiment Analysis (ABSA).
Return a JSON object with an 'aspects' array containing objects with `category` and `sentiment` for each category.
The sentiment should be one of the following: positive, neutral, negative.
**IMPORTANT:** Only return categories for which a sentiment is actually expressed in the text.
If no sentiment is expressed toward the mentioned categories in the text, return an empty array.
Categories without recognizable sentiment should **not** be included in the response."""

    return user_prompt


# Example call
text = "The battery life of this phone is amazing, but the camera quality is mediocre."
categories = ["battery life", "camera quality", "screen resolution"]
prompt = get_user_prompt(text, categories)
print(prompt)

Text: The battery life of this phone is amazing, but the camera quality is mediocre.
Categories: ['battery life', 'camera quality', 'screen resolution']

You are an expert in Aspect-Based Sentiment Analysis (ABSA).
Return a JSON object with an 'aspects' array containing objects with `category` and `sentiment` for each category.
The sentiment should be one of the following: positive, neutral, negative.
**IMPORTANT:** Only return categories for which a sentiment is actually expressed in the text.
If no sentiment is expressed toward the mentioned categories in the text, return an empty array.
Categories without recognizable sentiment should **not** be included in the response.


In [17]:
from pydantic import BaseModel
from enum import Enum
import json

# Sentiment is an enumeration with three possible values
class Sentiment(str, Enum):
    positive = "positive"
    neutral = "neutral"
    negative = "negative"

# Each aspect must consist of a category and a sentiment, where categories are strings and sentiment must be one of the three enum values.
class ABSAItem(BaseModel):
    category: str
    sentiment: Sentiment

# Response consists of a list of ABSAItems, i.e., aspects with category and sentiment
class ABSAResponse(BaseModel):
    aspects: list[ABSAItem]

def llm_text_to_json(user_prompt):
    completion = client.chat.completions.parse(
        temperature=0,
        model=LLM_REASONING,
        reasoning_effort="none",  # not needed for a classification task this simple (see 04_2_intro_structured_outputs.ipynb)
        messages=[
            {"role": "user", "content": user_prompt} # The prompt is passed as a user message
        ],
        response_format=ABSAResponse, # The output should be returned in the defined format
    )

    output = completion.choices[0].message.content # Extract the text from the LLM response

    structured_output = json.loads(output) # Parse the JSON string into a Python dictionary
    return structured_output


# Now let's define some examples that we can use in the UI
examples = [
    [
        "The food was delicious, but the service was very slow.",
        "Food, Service",
    ],
    [
        "The phone has a great camera, but the battery is weak.",
        "Camera, Battery, Design",
    ],
    [
        "The delivery arrived on time, but the packaging was damaged.",
        "Delivery, Packaging",
    ],
]

<details>
<summary><b>Show solution</b></summary>

```python
# Here you can create the Gradio app...
def analyze_sentiment_guided(text, categories):
    # First check if the inputs are valid
    # and return corresponding error messages if not.
    if not text.strip():
        return [], "Error, please enter a text."
    if not categories.strip():
        return [], "Error, please enter at least one category."

    categories_list = [c.strip() for c in categories.split(",") if c.strip()]
    if not categories_list:
        return [], "Error, please enter at least one category."


    # Create the user prompt and call the LLM to perform sentiment analysis
    user_prompt = get_user_prompt(text, categories_list)
    structured_output = llm_text_to_json(user_prompt)


    # Create table data
    table_data = []
    
    for item in structured_output["aspects"]:
        if item["sentiment"] == "positive":
            sentiment_text = "😊 Positive"
        elif item["sentiment"] == "negative":
            sentiment_text = "😞 Negative"
        else:
            sentiment_text = "😐 Neutral"
        
        table_data.append([item["category"], sentiment_text])

    if not table_data:
        return [], "Note: No category mentioned in the text."

    return table_data, ""

# Update the Gradio interface to use Table output
with gr.Blocks() as demo:
    # Of course, you can also use a normal Textbox here. Markdown is just for formatting.
    gr.Markdown("## Aspect-Based Sentiment Analysis (ABSA) with Guided JSON")
    
    error_text = gr.Text(
        label="Status",
        visible=True
    )

    with gr.Row():
        text_input = gr.Textbox(
            label="Enter text", placeholder="Enter a text...", lines=5
        )

    with gr.Row():
        category_list = gr.Textbox(
            label="Categories (comma-separated)", placeholder="e.g. Price, Quality, Delivery"
        )

    analyze_btn = gr.Button("Analyze sentiment")
    output_table = gr.Dataframe(
        headers=["Category", "Sentiment"],
        datatype=["str", "str"],
        label="ABSA Result",
    )

    analyze_btn.click(
        fn=analyze_sentiment_guided,
        inputs=[text_input, category_list],
        outputs=[output_table, error_text],
    )

    gr.Examples(
        examples=examples,
        inputs=[text_input, category_list],
        label="Examples",
    )

demo.launch()
```

</details>